# Plant Disease Assistant — Experiments and Results

This notebook is the reproducible results dashboard for the project. It explains the experimental protocol and reads current artifacts from `outputs/`; it does not retrain models or alter results. Missing experiments are reported explicitly, so the notebook remains usable while cross-validation or cross-domain evaluation is still running.

In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'configs' / 'project.yaml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the project checkout')

ROOT = find_project_root()
OUTPUTS = ROOT / 'outputs'
pd.options.display.max_colwidth = 80
print(f'Project root: {ROOT}')

Project root: /Users/fingerprint101/Documents/unipi/IS/plant-disease-assistant


## Experimental design

- **Primary data:** PlantSeg field photographs, using its official train/validation/test split and 115-class taxonomy.
- **Systems:** standalone disease-aware YOLO11n versus class-agnostic lesion YOLO11n followed by a baseline CNN, EfficientNetB0, or MobileNetV3-Large.
- **Model selection:** validation macro F1; the official PlantSeg test split remains a holdout.
- **Cross-validation:** three source-grouped folds over the combined official training and validation crops. Source URLs form groups to reduce near-duplicate leakage.
- **Cross-domain test:** the unchanged PlantSeg pipeline is evaluated on 21 explicitly mapped diseases in matched PlantSeg and PlantVillage test subsets.
- **Robustness:** six synthetic corruptions at five severity levels.
- **Explanation:** Grad-CAM is compared pixel-for-pixel with PlantSeg lesion masks.

PlantDoc is optional backup data and is not part of the required experiment set.

## Documented final-run snapshot

The project report records the following completed 100-epoch results. The live sections below prefer machine-readable artifacts when they are present.

| System | PlantSeg accuracy | Macro F1 | Coverage |
|---|---:|---:|---:|
| Standalone disease-aware YOLO | 61.37% | 53.32% | 86.48% |
| Lesion YOLO + baseline CNN | 41.32% | 29.04% | 100% |
| Lesion YOLO + EfficientNetB0 | 66.69% | 59.02% | 100% |
| Lesion YOLO + MobileNetV3-Large | **66.75%** | 58.17% | 100% |

Final lesion-YOLO localization reached 86.08% mAP@50 and 59.3% mAP@50–95; disease-aware YOLO reached 50.02% and 35.4%, respectively. Final MobileNetV3-Large Grad-CAM achieved 24.2% mean top-20% IoU, 41.5% mask-energy fraction, and a 58.8% pointing-game hit rate. In the full corruption run, severity-5 Gaussian blur was the worst condition: EfficientNetB0 accuracy fell from 67.3% to 29.7%, and MobileNetV3-Large fell from 66.5% to 30.5%. See `docs/preliminary_model_test.md` for the complete recorded tables and interpretation.

In [2]:
artifacts = {
    'Primary holdout': OUTPUTS / 'evaluation' / 'plantseg_test' / 'summary.json',
    'Cross-validation': OUTPUTS / 'cross_validation' / 'summary.json',
    'Cross-domain': OUTPUTS / 'evaluation' / 'cross_domain' / 'summary.json',
    'Grad-CAM': OUTPUTS / 'gradcam' / 'plantseg_test' / 'summary.json',
    'Robustness (full)': OUTPUTS / 'robustness' / 'plantseg_test_full' / 'summary.json',
    'Robustness (subset)': OUTPUTS / 'robustness' / 'plantseg_test' / 'summary.json',
}
artifact_status = pd.DataFrame([
    {'Experiment': name, 'Available': path.is_file(), 'Artifact': str(path.relative_to(ROOT))}
    for name, path in artifacts.items()
])
display(artifact_status.style.hide(axis='index'))

Experiment,Available,Artifact
Primary holdout,True,outputs/evaluation/plantseg_test/summary.json
Cross-validation,False,outputs/cross_validation/summary.json
Cross-domain,False,outputs/evaluation/cross_domain/summary.json
Grad-CAM,False,outputs/gradcam/plantseg_test/summary.json
Robustness (full),False,outputs/robustness/plantseg_test_full/summary.json
Robustness (subset),False,outputs/robustness/plantseg_test/summary.json


## Dataset coverage

In [3]:
metadata_files = {
    'PlantSeg training': ROOT / 'data/training/PlantSeg/Metadata.csv',
    'PlantSeg full test': ROOT / 'data/tests/PlantSeg/full/Metadata.csv',
    'PlantSeg mapped test': ROOT / 'data/tests/overlap/PlantSeg/Metadata.csv',
    'PlantVillage mapped test': ROOT / 'data/tests/overlap/PlantVillage/Metadata.csv',
}
coverage = []
for name, path in metadata_files.items():
    if path.is_file():
        frame = pd.read_csv(path)
        coverage.append({'Dataset view': name, 'Images': len(frame), 'Classes': frame['Index'].nunique()})
display(pd.DataFrame(coverage).style.hide(axis='index'))

Dataset view,Images,Classes
PlantSeg training,5367,115
PlantSeg full test,1561,114
PlantSeg mapped test,378,21
PlantVillage mapped test,6057,21


## Classifier training histories

In [4]:
model_names = ['baseline_cnn', 'efficientnet_b0', 'mobilenet_v3_large']
history_rows = []
best_rows = []
for model in model_names:
    path = OUTPUTS / 'classification' / model / 'history.json'
    if not path.is_file():
        continue
    history = json.loads(path.read_text())
    for epoch in history:
        history_rows.append({
            'model': model, 'epoch': epoch['epoch'],
            'train_macro_f1': epoch['train']['macro_f1'],
            'validation_macro_f1': epoch['validation']['macro_f1'],
        })
    best = max(history, key=lambda item: item['validation']['macro_f1'])
    best_rows.append({
        'Model': model, 'Best epoch': best['epoch'],
        'Validation macro F1': best['validation']['macro_f1'],
        'Validation accuracy': best['validation']['accuracy'],
    })
display(pd.DataFrame(best_rows).style.format({'Validation macro F1': '{:.3f}', 'Validation accuracy': '{:.3f}'}).hide(axis='index'))
if history_rows:
    history_df = pd.DataFrame(history_rows)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
    for model, group in history_df.groupby('model'):
        axes[0].plot(group['epoch'], group['train_macro_f1'], label=model)
        axes[1].plot(group['epoch'], group['validation_macro_f1'], label=model)
    axes[0].set_title('Training macro F1'); axes[1].set_title('Validation macro F1')
    for axis in axes:
        axis.set_xlabel('Epoch'); axis.set_ylabel('Macro F1'); axis.grid(alpha=.25)
    axes[1].legend(); plt.tight_layout(); plt.show()

Model,Best epoch,Validation macro F1,Validation accuracy
baseline_cnn,10,0.066,0.173
efficientnet_b0,10,0.564,0.677
mobilenet_v3_large,9,0.542,0.669


/var/folders/7j/0dwm2zx51qs8c8dkjt50vpth0000gn/T/ipykernel_70273/1129181408.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  axes[1].legend(); plt.tight_layout(); plt.show()


## Primary PlantSeg holdout

The primary comparison measures end-to-end image classification on PlantSeg's untouched official test split. A missed lesion detection causes the two-stage system to classify the full image; a missed standalone-YOLO detection counts as an incorrect prediction.

In [5]:
primary_path = artifacts['Primary holdout']
if primary_path.is_file():
    primary = json.loads(primary_path.read_text())
    rows = []
    standalone = primary['standalone_yolo']['classification']
    rows.append({'System': 'standalone_yolo', **{key: standalone.get(key) for key in ['accuracy', 'macro_f1', 'balanced_accuracy', 'coverage']}})
    for model, values in primary['classification'].items():
        rows.append({'System': f'lesion_yolo + {model}', **{key: values.get(key) for key in ['accuracy', 'macro_f1', 'balanced_accuracy']}, 'coverage': 1.0})
    primary_df = pd.DataFrame(rows)
    display(primary_df.style.format({key: '{:.3f}' for key in ['accuracy', 'macro_f1', 'balanced_accuracy', 'coverage']}).hide(axis='index'))
    localization = pd.DataFrame([
        {'Detector': 'class-agnostic lesion YOLO', **primary['localization']},
        {'Detector': 'disease-aware YOLO', **primary['standalone_yolo']['localization']},
    ]).drop(columns=['speed_ms_per_image'])
    display(localization.style.format(precision=3).hide(axis='index'))
    current_epochs = {row['Model']: int(pd.read_json(OUTPUTS / 'classification' / row['Model'] / 'history.json')['epoch'].max()) for row in best_rows}
    recorded_epochs = {name: details['epoch'] for name, details in primary['classifier_checkpoints'].items()}
    if any(current_epochs.get(name) != epoch for name, epoch in recorded_epochs.items()):
        display(Markdown('> **Provenance warning:** this holdout summary references older classifier checkpoints than the current training histories. Rerun `make evaluate` before using it as the final result.'))
else:
    display(Markdown('> Primary evaluation is not available. Run `make evaluate`.'))

System,accuracy,macro_f1,balanced_accuracy,coverage
standalone_yolo,0.229,0.138,0.121,0.416
lesion_yolo + baseline_cnn,0.167,0.073,0.084,1.000
lesion_yolo + efficientnet_b0,0.691,0.609,0.610,1.000
lesion_yolo + mobilenet_v3_large,0.681,0.613,0.610,1.000


Detector,precision,recall,map50,map50_95
class-agnostic lesion YOLO,0.756,0.739,0.776,0.462
disease-aware YOLO,0.540,0.165,0.154,0.109


> **Provenance warning:** this holdout summary references older classifier checkpoints than the current training histories. Rerun `make evaluate` before using it as the final result.

## Source-grouped cross-validation

Three-fold cross-validation estimates variation across development-data splits. These scores support model comparison; they do not replace the untouched official test evaluation.

In [6]:
cv_path = artifacts['Cross-validation']
if cv_path.is_file():
    cv = json.loads(cv_path.read_text())
    cv_rows = []
    for model, values in cv['models'].items():
        cv_rows.append({
            'Model': model, 'Mean accuracy': values['mean_accuracy'], 'SD accuracy': values['std_accuracy'],
            'Mean macro F1': values['mean_macro_f1'], 'SD macro F1': values['std_macro_f1'],
            'Pooled macro F1': values['pooled_macro_f1'],
        })
    display(pd.DataFrame(cv_rows).style.format(precision=3).hide(axis='index'))
else:
    display(Markdown('> Cross-validation results are pending. They will appear here after `outputs/cross_validation/summary.json` is produced.'))

> Cross-validation results are pending. They will appear here after `outputs/cross_validation/summary.json` is produced.

## PlantVillage cross-domain evaluation

Both domains contain only the same 21 mapped disease labels. Every image passes through the same lesion detector, crop policy, classifier checkpoints, and preprocessing. The reported change is **PlantVillage minus matched PlantSeg**, so a negative value denotes degradation under domain shift.

In [7]:
cross_path = artifacts['Cross-domain']
if cross_path.is_file():
    cross = json.loads(cross_path.read_text())
    cross_rows = []
    for model in cross['classifier_checkpoints']:
        source = cross['domains']['plantseg_overlap']['classification'][model]
        target = cross['domains']['plantvillage']['classification'][model]
        cross_rows.append({
            'Model': model, 'PlantSeg accuracy': source['accuracy'], 'PlantVillage accuracy': target['accuracy'],
            'Accuracy change': target['accuracy'] - source['accuracy'],
            'PlantSeg macro F1': source['macro_f1'], 'PlantVillage macro F1': target['macro_f1'],
            'Macro F1 change': target['macro_f1'] - source['macro_f1'],
            'PlantVillage ECE': target['expected_calibration_error'],
        })
    display(pd.DataFrame(cross_rows).style.format(precision=3).hide(axis='index'))
    image_path = cross_path.parent / 'domain_comparison.png'
    if image_path.is_file(): display(Markdown(f'![Domain comparison]({image_path.as_posix()})'))
else:
    display(Markdown('> Cross-domain results are not available. Run `make evaluate-cross-domain`.'))

> Cross-domain results are not available. Run `make evaluate-cross-domain`.

## Robustness under synthetic corruption

In [8]:
robust_path = artifacts['Robustness (full)'] if artifacts['Robustness (full)'].is_file() else artifacts['Robustness (subset)']
if robust_path.is_file():
    robust = json.loads(robust_path.read_text())
    display(pd.json_normalize(robust).T.rename(columns={0: 'Value'}))
    plot_path = robust_path.parent / 'accuracy_vs_severity.png'
    if plot_path.is_file(): display(Markdown(f'![Accuracy versus severity]({plot_path.as_posix()})'))
else:
    display(Markdown('> Robustness artifacts are not present in this checkout. Run `make robustness`; documented final results remain in `docs/preliminary_model_test.md`.'))

> Robustness artifacts are not present in this checkout. Run `make robustness`; documented final results remain in `docs/preliminary_model_test.md`.

## Grad-CAM localization against lesion masks

In [9]:
gradcam_path = artifacts['Grad-CAM']
if gradcam_path.is_file():
    gradcam = json.loads(gradcam_path.read_text())
    display(pd.json_normalize(gradcam).T.rename(columns={0: 'Value'}))
    figures = sorted((gradcam_path.parent / 'figures').glob('*.png'))[:6]
    if figures:
        fig, axes = plt.subplots(2, 3, figsize=(15, 9))
        for axis, path in zip(axes.flat, figures):
            axis.imshow(plt.imread(path)); axis.set_title(path.stem); axis.axis('off')
        plt.tight_layout(); plt.show()
else:
    display(Markdown('> Grad-CAM artifacts are not present in this checkout. Run `make gradcam`; documented final results remain in `docs/preliminary_model_test.md`.'))

> Grad-CAM artifacts are not present in this checkout. Run `make gradcam`; documented final results remain in `docs/preliminary_model_test.md`.

## Interpretation checklist

When preparing the final report:

1. Select the classifier using validation and cross-validation macro F1, not test accuracy.
2. Rerun stale holdout artifacts once after checkpoints are fixed.
3. Report fold mean ± standard deviation alongside the official holdout result.
4. Treat PlantVillage performance as evidence about domain shift, not as an interchangeable extension of PlantSeg.
5. Discuss calibration together with accuracy: confident mistakes are particularly important for an assistant.
6. Treat Grad-CAM as an explanation signal, not a verified segmentation method.